In [1]:
!pip install pymupdf pymupdf4llm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 74.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.3/84.3 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 93.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 94.3 MB/s eta 0:00:00


In [2]:
import os
import json
import requests
import io
import fitz  # PyMuPDF
import pymupdf4llm  # Thư viện chuyển PDF sang Markdown của PyMuPDF
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm.auto import tqdm # Dùng auto để thanh trình duyệt hiển thị đẹp hơn trên Kaggle

In [3]:
# --- CẤU HÌNH CHO KAGGLE ---
INPUT_FILE = "/kaggle/input/datasets/tranphungdinh/3-strict-filter/final_cleaned_data.jsonl" 
OUTPUT_DIR = "/kaggle/working/parquet_output" 
BATCH_SIZE = 50             

# Tạo thư mục output nếu chưa có
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Schema của file Parquet
schema = pa.schema([
    ('paper_id', pa.string()),
    ('title', pa.string()),
    ('source_url', pa.string()),
    ('full_text', pa.string())
])

In [4]:
def get_pdf_url(doc):
    """Trích xuất URL tải PDF ưu tiên lấy link Open Access trực tiếp"""
    # 1. Ưu tiên cao nhất: Lấy link PDF trực tiếp từ field openAccessPdf
    oa_info = doc.get("openAccessPdf")
    if oa_info and isinstance(oa_info, dict) and oa_info.get("url"):
        return oa_info.get("url")
    
    externalsid = doc.get("externalsid", {})
    if not externalsid: return None
    
    # 2. Fallback 1: ArXiv
    arxiv_id = externalsid.get("arxiv")
    if arxiv_id:
        clean_id = arxiv_id.replace("arXiv:", "").strip()
        return f"https://arxiv.org/pdf/{clean_id}.pdf"
        
    # 3. Fallback 2: ACL
    acl_url = externalsid.get("acl")
    if acl_url:
        if not acl_url.endswith(".pdf"):
            return f"{acl_url}.pdf"
        return acl_url
        
    return None


def extract_text_from_pdf_stream(pdf_url):
    """Tải PDF vào RAM và trích xuất text dưới dạng Markdown (TẮT OCR)"""
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Accept': 'application/pdf,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8'
    }
    
    try:
        response = requests.get(pdf_url, stream=True, timeout=30, headers=headers)
        response.raise_for_status()
        
        # Check xem có phải file PDF thật không
        content_type = response.headers.get('Content-Type', '')
        if 'application/pdf' not in content_type and 'application/octet-stream' not in content_type:
            return None
            
        pdf_stream = io.BytesIO(response.content)
        doc = fitz.open(stream=pdf_stream, filetype="pdf")
        
        # TRÍCH XUẤT MARKDOWN, TẮT HOÀN TOÀN OCR
        markdown_text = pymupdf4llm.to_markdown(
            doc,
            use_ocr=False,       # Chặn đứng Tesseract OCR (trả về rỗng ở những trang chỉ toàn ảnh)
            write_images=False   # Bỏ qua không xử lý ảnh minh họa
        )
                
        doc.close()
        pdf_stream.close()
        del response
        
        return markdown_text
        
    except Exception:
        return None
       

In [5]:
def main():
    if not os.path.exists(INPUT_FILE):
        print(f"❌ Không tìm thấy file {INPUT_FILE}")
        print("💡 Hãy kiểm tra lại đường dẫn trong thư mục /kaggle/input/ bên cột phải!")
        return

    # Đếm số dòng để hiển thị thanh tiến trình (tqdm)
    with open(INPUT_FILE, 'r', encoding='utf-8') as f:
        total_lines = sum(1 for _ in f)

    output_filepath = os.path.join(OUTPUT_DIR, "all_papers.parquet")

    print(f"🚀 Bắt đầu xử lý TOÀN BỘ dataset")
    print(f"   📍 Tổng số dòng trong JSONL: {total_lines}")
    print(f"   💾 Output: {output_filepath}")
    print()

    batch_records = []
    total_processed = 0
    total_success = 0
    
    writer = pq.ParquetWriter(output_filepath, schema)

    with open(INPUT_FILE, 'r', encoding='utf-8') as f:
        for line in tqdm(f, total=total_lines, desc="Processing All Papers"):
            try:
                doc = json.loads(line)
                paper_id = str(doc.get("paper_id", ""))
                title = doc.get("title", "")
                
                pdf_url = get_pdf_url(doc)
                if not pdf_url:
                    continue
                    
                total_processed += 1
                
                # Gọi hàm extract (đã tích hợp pymupdf4llm ở bước trước)
                full_text = extract_text_from_pdf_stream(pdf_url)
                
                if full_text and len(full_text.strip()) > 500:
                    batch_records.append({
                        "paper_id": paper_id,
                        "title": title,
                        "source_url": pdf_url,
                        "full_text": full_text
                    })
                    total_success += 1
                
                # Ghi vào Parquet và xả RAM khi đủ batch
                if len(batch_records) >= BATCH_SIZE:
                    table = pa.Table.from_pylist(batch_records, schema=schema)
                    writer.write_table(table)
                    batch_records = [] 
                    
            except Exception as e:
                # Bỏ qua lỗi (timeout, pdf hỏng...) để script tiếp tục chạy
                continue

    # Ghi nốt những bài còn sót lại trong batch cuối cùng
    if len(batch_records) > 0:
        table = pa.Table.from_pylist(batch_records, schema=schema)
        writer.write_table(table)
    
    writer.close()
    
    print("\n🎉 HOÀN TẤT QUÁ TRÌNH CRAWL!")
    print(f"📊 Đã quét và tìm thấy link PDF cho: {total_processed} papers")
    print(f"✅ Đã chuyển Markdown và lưu thành công: {total_success} papers")
    print(f"📂 Output duy nhất được lưu tại: {output_filepath}")

if __name__ == "__main__":
    main()

🚀 Bắt đầu xử lý TOÀN BỘ dataset
   📍 Tổng số dòng trong JSONL: 13013
   💾 Output: /kaggle/working/parquet_output/all_papers.parquet



Processing All Papers:   0%|          | 0/13013 [00:00<?, ?it/s]

MuPDF error: format error: cmsOpenProfileFromMem failed

MuPDF error: format error: cmsOpenProfileFromMem failed

MuPDF error: format error: cmsOpenProfileFromMem failed

MuPDF error: format error: cmsOpenProfileFromMem failed

MuPDF error: syntax error: unknown keyword: 'pagesize'

MuPDF error: syntax error: unknown keyword: 'width'

MuPDF error: syntax error: unknown keyword: '597.50787pt'

MuPDF error: syntax error: unknown keyword: 'height'

MuPDF error: syntax error: unknown keyword: '845.04675pt'

MuPDF error: syntax error: unknown keyword: 'pagesize'

MuPDF error: syntax error: unknown keyword: 'width'

MuPDF error: syntax error: unknown keyword: '597.50787pt'

MuPDF error: syntax error: unknown keyword: 'height'

MuPDF error: syntax error: unknown keyword: '845.04675pt'

MuPDF error: syntax error: unknown keyword: 'pagesize'

MuPDF error: syntax error: unknown keyword: 'width'

MuPDF error: syntax error: unknown keyword: '597.50787pt'

MuPDF error: syntax error: unknown keyword